# 고객 이탈 데이터 전처리 분석 계획

## 데이터 점검 결과 요약

- 파일명: `customer_churn_dummy.csv`
- 지정 데이터프레임 변수명: `df`
- 데이터 크기: 2,000행 × 13열
- 주요 열: `고객ID`, `성별`, `나이`, `가입채널`, `요금제`, `가입일`, `가입경과월`, `최근30일_로그인수`, `월평균사용시간`, `고객센터문의수`, `최근3개월_결제실패`, `이탈여부`, `이탈일`
- 자료형 요약: 수치형 6개(`나이`, `가입경과월`, `최근30일_로그인수`, `월평균사용시간`, `고객센터문의수`, `최근3개월_결제실패`), 문자형 7개(`고객ID`, `성별`, `가입채널`, `요금제`, `가입일`, `이탈여부`, `이탈일`)
- 결측치: `이탈일`에 1,668건(83.4%)이 있으며, 이는 `이탈여부=N` 고객의 정상적인 미이탈 상태와 대응됩니다. 그 외 열의 결측치는 없습니다.
- 타깃 분포: `이탈여부`는 N 1,668건, Y 332건이며, 이탈 비율은 16.6%입니다.
- 중복 및 논리 점검: `고객ID` 중복 0건, `이탈여부=N인데 이탈일 존재` 0건, `이탈여부=Y인데 이탈일 결측` 0건, `이탈일이 가입일보다 빠름` 0건입니다.
- 날짜 점검: 가입일: 날짜 변환 오류 0건, 범위 2023-06-02 ~ 2026-04-30; 이탈일: 날짜 변환 오류 0건, 범위 2024-12-08 ~ 2026-05-26입니다.
- IQR 기준 이상치 후보: `최근30일_로그인수` 5건, `월평균사용시간` 36건, `고객센터문의수` 62건, `최근3개월_결제실패` 433건입니다. `최근3개월_결제실패`는 0이 많은 카운트형 변수라 단순 제거보다 의미 확인이 필요합니다.
- 전처리 핵심 방향: 날짜형 변환, 타깃과 이탈일의 누수 관리, 범주형 인코딩 기준 정리, 수치형 이상치 처리 원칙 수립, 모델링용 학습 데이터셋 분리입니다.


### 1단계. 데이터 불러오기와 변수명 고정

- 지시: customer_churn_dummy.csv 파일을 `df`라는 데이터프레임으로 불러오고, `고객ID`가 고객 단위 고유 식별자인지 다시 확인하세요.
- 이유: 이후 모든 전처리 단계가 같은 데이터프레임을 기준으로 이어지도록 하기 위해서입니다.

In [1]:
# 이 셀은 필요한 라이브러리를 불러옵니다.
import pandas as pd

In [2]:
# 이 셀은 csv 파일을 읽어 df 데이터프레임으로 저장합니다.
df = pd.read_csv(filepath_or_buffer="customer_churn_dummy.csv")

df.head()

,고객ID,성별,나이,가입채널,요금제,가입일,가입경과월,최근30일_로그인수,월평균사용시간,고객센터문의수,최근3개월_결제실패,이탈여부,이탈일
0,C1000,여,26,오프라인매장,스탠다드,2024-11-07,19,3,9.9,1,1,Y,2026-04-10
1,C1001,남,49,제휴이벤트,베이직,2025-06-07,11,18,9.7,1,0,N,NaN
2,C1002,여,24,오프라인매장,스탠다드,2026-01-16,4,18,28.3,0,0,N,NaN
3,C1003,여,35,지인추천,프리미엄,2024-07-23,22,4,3.8,1,0,Y,2024-12-26
4,C1004,남,31,소셜미디어,베이직,2024-07-14,22,13,6.7,2,0,N,NaN


In [3]:
# 이 셀은 고객ID가 고객 단위 고유 식별자인지 확인합니다.
# 전체 행 개수와 고객ID 고유값 개수가 같으면 고유 식별자입니다.
id_check = pd.DataFrame(
    data={
        "전체_행_개수": [df.shape[0]],
        "고객ID_고유값_개수": [df["고객ID"].nunique()],
        "고객ID_중복_개수": [df["고객ID"].duplicated().sum()],
        "고유_식별자_여부": [df["고객ID"].is_unique],
    }
)

id_check

,전체_행_개수,고객ID_고유값_개수,고객ID_중복_개수,고유_식별자_여부
0,2000,2000,0,True


In [4]:
# 이 셀은 중복된 고객ID가 있는 경우 해당 행을 확인합니다.
df[df["고객ID"].duplicated(keep=False)].sort_values(by="고객ID")

,고객ID,성별,나이,가입채널,요금제,가입일,가입경과월,최근30일_로그인수,월평균사용시간,고객센터문의수,최근3개월_결제실패,이탈여부,이탈일


### 2단계. 열 역할 분류

- 지시: `df`의 열을 식별자(`고객ID`), 타깃(`이탈여부`), 날짜형(`가입일`, `이탈일`), 범주형(`성별`, `가입채널`, `요금제`), 수치형(`나이`, `가입경과월`, `최근30일_로그인수`, `월평균사용시간`, `고객센터문의수`, `최근3개월_결제실패`)으로 분류해 전처리 기준표를 만드세요.
- 이유: 열의 역할에 따라 결측치 처리, 변환, 인코딩 방식이 달라지기 때문입니다.

In [ ]:
# 위 '지시'를 Python Code Assistant에 입력해 받은 코드를 여기에 붙여넣고 실행하세요.

### 3단계. 날짜형 변환과 논리 검증

- 지시: `df`의 `가입일`과 `이탈일`을 날짜형으로 변환하고, `이탈여부`와 `이탈일`의 대응 관계, `이탈일`이 `가입일`보다 빠른 사례, 날짜 변환 실패 사례를 점검하세요.
- 이유: 날짜 오류는 가입 기간 계산과 이탈 분석 기준을 왜곡할 수 있기 때문입니다.

In [ ]:
# 위 '지시'를 Python Code Assistant에 입력해 받은 코드를 여기에 붙여넣고 실행하세요.

### 4단계. 결측치 처리 원칙 수립

- 지시: `df`에서 `이탈일` 결측은 `이탈여부=N`인 미이탈 고객의 구조적 결측으로 유지하되, 모델 학습용 데이터셋에서는 `이탈일`을 제외하거나 이탈 후에만 알 수 있는 정보로 별도 관리하세요.
- 이유: `이탈일`은 타깃 이후 정보라 이탈 예측 전처리에서 데이터 누수를 만들 수 있기 때문입니다.

In [ ]:
# 위 '지시'를 Python Code Assistant에 입력해 받은 코드를 여기에 붙여넣고 실행하세요.

### 5단계. 타깃 변수 정리

- 지시: `df`의 `이탈여부`를 분석용 타깃 변수로 정리하고, Y와 N의 의미를 명확히 기록한 뒤 모델링용 별도 타깃 열을 만들 준비를 하세요.
- 이유: 문자형 타깃을 일관된 기준으로 관리해야 이후 시각화와 모델링 단계에서 오류가 줄어들기 때문입니다.

In [ ]:
# 위 '지시'를 Python Code Assistant에 입력해 받은 코드를 여기에 붙여넣고 실행하세요.

### 6단계. 범주형 변수 값 점검과 정제

- 지시: `df`의 `성별`, `가입채널`, `요금제`, `이탈여부`에서 고유값, 빈도, 오탈자·공백·희소 범주 여부를 확인하고 필요한 경우 표준화 기준을 정하세요.
- 이유: 범주 값이 불일치하면 그룹별 집계와 인코딩 결과가 왜곡될 수 있기 때문입니다.

In [ ]:
# 위 '지시'를 Python Code Assistant에 입력해 받은 코드를 여기에 붙여넣고 실행하세요.

### 7단계. 수치형 변수 범위와 이상치 점검

- 지시: `df`의 `나이`, `가입경과월`, `최근30일_로그인수`, `월평균사용시간`, `고객센터문의수`, `최근3개월_결제실패`에 대해 값의 범위, 음수 여부, 극단값 후보를 확인하고 변수별 처리 기준을 정하세요.
- 이유: 고객 행동 변수의 극단값은 실제 우수·위험 고객 신호일 수도 있어 무조건 제거하면 안 되기 때문입니다.

In [ ]:
# 위 '지시'를 Python Code Assistant에 입력해 받은 코드를 여기에 붙여넣고 실행하세요.

### 8단계. 가입기간 관련 파생 점검

- 지시: `df`의 `가입일`과 기준일을 이용해 계산한 가입 경과 기간이 기존 `가입경과월`과 크게 어긋나는지 확인하고, 불일치가 크면 사용할 가입기간 기준을 하나로 정하세요.
- 이유: 동일한 의미의 기간 변수가 서로 다르면 고객 생애주기 분석 결과가 달라질 수 있기 때문입니다.

In [ ]:
# 위 '지시'를 Python Code Assistant에 입력해 받은 코드를 여기에 붙여넣고 실행하세요.

### 9단계. 학습용 입력 변수 후보 정리

- 지시: `df`에서 `고객ID`, `이탈일`처럼 예측 입력으로 부적절하거나 누수 가능성이 있는 열을 제외하고, `성별`, `나이`, `가입채널`, `요금제`, `가입경과월`, `최근30일_로그인수`, `월평균사용시간`, `고객센터문의수`, `최근3개월_결제실패`을 입력 변수 후보로 정리하세요.
- 이유: 전처리 단계에서 사용할 피처 범위를 먼저 고정해야 이후 분석이 일관되게 진행되기 때문입니다.

In [ ]:
# 위 '지시'를 Python Code Assistant에 입력해 받은 코드를 여기에 붙여넣고 실행하세요.

### 10단계. 인코딩과 스케일링 계획 수립

- 지시: `df`의 범주형 입력 변수는 모델링에 사용할 수 있도록 인코딩 계획을 세우고, 수치형 입력 변수는 거리 기반 모델 사용 가능성을 고려해 스케일링 적용 여부를 정하세요.
- 이유: 범주형과 수치형 변환 기준을 분리해야 다양한 모델에 같은 전처리 원칙을 적용할 수 있기 때문입니다.

In [ ]:
# 위 '지시'를 Python Code Assistant에 입력해 받은 코드를 여기에 붙여넣고 실행하세요.

### 11단계. 전처리 결과 데이터셋 저장 기준 정리

- 지시: 위 기준을 적용한 전처리 결과를 원본 `df`와 구분되는 모델링용 데이터프레임으로 만들고, 타깃 분포와 행 수가 원본과 의도대로 유지되는지 확인한 뒤 저장 계획을 세우세요.
- 이유: 원본 보존과 전처리본 관리를 분리해야 재현 가능한 분석이 가능하기 때문입니다.

In [ ]:
# 위 '지시'를 Python Code Assistant에 입력해 받은 코드를 여기에 붙여넣고 실행하세요.